# 항공 지연 EDA — Google Colab

**이 노트북 하나로 실행합니다. GitHub clone이나 별도 코드 ZIP이 필요 없습니다.**

1. Colab에 이 `.ipynb` 파일을 업로드합니다.
2. CPU 런타임에서 셀을 위에서부터 실행합니다. GPU는 필요하지 않습니다.
3. CSV 입력 방식은 Google Drive(권장) 또는 직접 업로드입니다. 약 205MB 파일이므로 Drive에 먼저 올리는 방법을 권장합니다.
4. 결과를 확인한 뒤 마지막 셀에서 결과 ZIP을 다운로드합니다.

분석 대상: 지연 원인별 특성 / 동일 기체 운항 연결 / 공항 운항량·혼잡 징후.
기본 설정은 제공된 **2026년 1월 BTS Reporting Carrier CSV**에 맞췄습니다. 원본은 변경하지 않습니다.
메모리가 부족하면 높은 메모리 런타임 또는 더 작은 기간의 입력을 사용하세요. 한 파일을 메모리에서 처리합니다.

## 1. 실행 환경 준비

In [ ]:
import sys, subprocess
assert sys.version_info >= (3, 11), 'Python 3.11 이상 런타임이 필요합니다.'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pandas>=2.2,<4', 'numpy>=1.26,<3', 'pyarrow>=15', 'matplotlib>=3.8', 'tzdata'])
print('설치 완료. 설치 과정에서 런타임 재시작 안내가 나오면 재시작 후 다음 셀부터 실행하세요.')

## 2. CSV 선택

Drive 방식은 **DRIVE_CSV_PATH**를 실제 파일 위치로 바꿉니다. 직접 업로드는 브라우저에서 CSV를 선택합니다. Colab은 PC의 다운로드 폴더에 직접 접근할 수 없습니다.

In [ ]:
from pathlib import Path
import json, hashlib, shutil, subprocess, sys, urllib.request
from datetime import datetime, timezone

INPUT_MODE = "drive" #@param ["drive", "upload"]
DRIVE_CSV_PATH = "/content/drive/MyDrive/T_ONTIME_REPORTING.csv" #@param {type:"string"}
ROOT = Path('/content/flight_eda_project')
ROOT.mkdir(parents=True, exist_ok=True)

if INPUT_MODE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    INPUT = Path(DRIVE_CSV_PATH)
elif INPUT_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    selected = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(selected) != 1:
        raise ValueError('CSV 파일을 정확히 하나 선택하세요.')
    INPUT = Path(selected[0]).resolve()
    del uploaded
else:
    raise ValueError('INPUT_MODE는 drive 또는 upload이어야 합니다.')
if not INPUT.is_file():
    raise FileNotFoundError(f'CSV 경로를 확인하세요: {INPUT}')
print(INPUT.name, f'{INPUT.stat().st_size / 1_000_000:.1f} MB')

## 3. 공통 분석 코드 준비

아래 셀에 전처리·집계·검증용 코드를 포함했습니다. GitHub 권한과 무관하게 실행됩니다. 코드를 수정하려면 이 셀의 `SOURCE_FILES`를 펼쳐 확인하세요.

In [ ]:
SOURCE_FILES = {'src/flight_eda/airport_stats.py': '"""Airport x UTC hour. Delay denominators exclude cancellations/diversions."""\nimport pandas as pd\n\n\ndef aggregate_airport_hourly(flights, threshold=15):\n    parts = []\n    regular = flights.cancelled.eq(0) & flights.diverted.eq(0)\n    for direction, airport, scheduled, actual, delay, taxi, valid in [\n        ("dep", "origin", "scheduled_dep_utc", "actual_dep_utc", "dep_delay", "taxi_out", "actual_dep_valid"),\n        ("arr", "dest", "scheduled_arr_utc", "actual_arr_utc", "arr_delay", "taxi_in", "actual_arr_valid")]:\n        f = pd.DataFrame({"airport": flights[airport], "hour_utc": flights[scheduled].dt.floor("h"),\n                          "delay": flights[delay].where(regular), "taxi": flights[taxi].where(regular),\n                          "cancelled": flights.cancelled, "diverted": flights.diverted})\n        # Unvalidated scheduled arrivals must not create apparently precise bins.\n        if direction == "arr":\n            f.loc[~flights.schedule_time_valid, "hour_utc"] = pd.NaT\n        f["delayed"] = f.delay.ge(threshold).astype(float).where(f.delay.notna())\n        groups = f.groupby(["airport", "hour_utc"])\n        grouped = groups.agg(\n            scheduled=("cancelled", "size"), delay_valid=("delay", "count"),\n            mean_delay=("delay", "mean"), median_delay=("delay", "median"),\n            delay_rate=("delayed", "mean"),\n            mean_taxi=("taxi", "mean"), taxi_valid=("taxi", "count"),\n            cancelled=("cancelled", "sum"), diverted=("diverted", "sum"))\n        grouped["p90_delay"] = groups.delay.quantile(.9)\n        grouped["cancel_rate"] = grouped.cancelled / grouped.scheduled\n        grouped = grouped.add_prefix(direction + "_")\n        # Actual traffic counts only regular validated events, explicitly named.\n        events = flights.loc[regular & flights[valid], [airport, actual]].copy()\n        events.columns = ["airport", "hour_utc"]\n        events["hour_utc"] = events.hour_utc.dt.floor("h")\n        counts = events.groupby(["airport", "hour_utc"]).size().rename(direction + "_actual_regular")\n        parts.extend([grouped, counts])\n    result = pd.concat(parts, axis=1)\n    count_cols = [c for c in result if any(c.endswith(s) for s in ["_scheduled", "_delay_valid", "_taxi_valid", "_cancelled", "_diverted", "_actual_regular"])]\n    result[count_cols] = result[count_cols].fillna(0).astype(int)\n    return result.reset_index()\n', 'src/flight_eda/cli.py': 'from pathlib import Path\nimport argparse\nimport importlib.metadata\nimport json\nimport subprocess\nimport tomllib\nimport pandas as pd\nfrom .preprocess import read_flights, add_times, diversion_events, file_hash, CAUSES\nfrom .delay_causes import summarize_causes\nfrom .connections import build_connections, recovery_summary\nfrom .airport_stats import aggregate_airport_hourly\nfrom .plots import make_figures\n\n\ndef run(input_path, airports, output, config):\n    output = Path(output)\n    if output.exists() and any(output.iterdir()):\n        raise ValueError("Output directory is not empty. Choose a new run directory to preserve previous results.")\n    output.mkdir(parents=True, exist_ok=True)\n    settings = tomllib.loads(Path(config).read_text(encoding="utf-8"))["analysis"]\n    print("Reading BTS CSV...", flush=True)\n    flights, duplicates = read_flights(input_path)\n    print(f"Reconstructing times for {len(flights):,} flights...", flush=True)\n    flights = add_times(flights, airports)\n    print("Building observed aircraft sequences...", flush=True)\n    connections = build_connections(flights, settings["max_connection_gap_hours"])\n    print("Aggregating causes and airport hours...", flush=True)\n    causes = summarize_causes(flights)\n    hourly = aggregate_airport_hourly(flights, settings["delay_threshold_min"])\n    events = diversion_events(flights)\n    for name, frame in [("flights", flights), ("flight_connections", connections), ("airport_hourly", hourly), ("flight_diversions", events)]:\n        frame.to_parquet(output / f"{name}.parquet", index=False)\n    causes.to_csv(output / "delay_causes.csv", index=False)\n    for by, name in [(["op_unique_carrier"], "carrier"), (["origin"], "airport"), (["scheduled_dep_hour"], "hour"), (["month"], "month")]:\n        if set(by).issubset(flights.columns):\n            summarize_causes(flights, by).to_csv(output / f"delay_causes_by_{name}.csv", index=False)\n    recovery_summary(connections).to_csv(output / "connection_recovery.csv", index=False)\n    flights.loc[flights.cancelled.eq(1)].groupby("cancellation_code", dropna=False).size().rename("flights").to_csv(output / "cancellations.csv")\n    sensitivity = []\n    for hours in [12, 24, 48]:\n        c = build_connections(flights, hours)\n        sensitivity.append({"max_gap_hours": hours, "valid_links": int(c.link_status.eq("valid").sum())})\n    pd.DataFrame(sensitivity).to_csv(output / "connection_sensitivity.csv", index=False)\n    eligible = flights.cancelled.eq(0) & flights.diverted.eq(0) & flights.arr_delay.ge(15)\n    complete = eligible & flights[CAUSES].notna().all(axis=1)\n    residual = flights.loc[complete, CAUSES].sum(axis=1) - flights.loc[complete, "arr_delay"]\n    valid = connections.loc[connections.link_status.eq("valid")]\n    quality = {\n        "rows": len(flights), "exact_duplicates_removed": duplicates,\n        "natural_key_conflict_rows": int(flights.key_conflict.sum()),\n        "date_min": str(flights.flight_date.min().date()), "date_max": str(flights.flight_date.max().date()),\n        "invalid_flight_date_rows": int(flights.flight_date.isna().sum()),\n        "airports": len(set(flights.origin.dropna()) | set(flights.dest.dropna())),\n        "cancelled": int(flights.cancelled.eq(1).sum()), "diverted": int(flights.diverted.eq(1).sum()),\n        "missing_tail": int(flights.tail_num.isna().sum()),\n        "missing_timezone_airports": sorted(set(flights.loc[flights.origin_timezone.isna(), "origin"].dropna()) | set(flights.loc[flights.dest_timezone.isna(), "dest"].dropna())),\n        "time_quality": flights.time_quality_flag.value_counts().to_dict(),\n        "link_status": connections.link_status.value_counts().to_dict(),\n        "cause_eligible_flights": int(eligible.sum()), "cause_complete_rows": int(complete.sum()),\n        "cause_sum_mismatch_over_1_min": int(residual.abs().gt(1).sum()),\n        "valid_link_pearson_correlation": float(valid[["prev_arr_delay_min", "next_dep_delay_min"]].corr().iloc[0, 1]) if len(valid) > 1 else None,\n        "excluded_from_scheduled_dep_hours": int(flights.scheduled_dep_utc.isna().sum()),\n        "excluded_from_scheduled_arr_hours": int((flights.scheduled_arr_utc.isna() | ~flights.schedule_time_valid).sum()),\n        "column_missing_counts": flights.isna().sum().to_dict(),\n    }\n    (output / "quality_report.json").write_text(json.dumps(quality, indent=2, ensure_ascii=False, default=int), encoding="utf-8")\n    try:\n        revision = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()\n    except (OSError, subprocess.CalledProcessError):\n        revision = "uncommitted-local-build"\n    source_dir = Path(__file__).parent\n    manifest = {"input_name": Path(input_path).name, "input_sha256": file_hash(input_path),\n                "airports_sha256": file_hash(airports), "config": settings, "git_revision": revision,\n                "code_sha256": {p.name: file_hash(p) for p in sorted(source_dir.glob("*.py"))},\n                "packages": {p: importlib.metadata.version(p) for p in ["pandas", "numpy", "pyarrow", "matplotlib", "tzdata"]}}\n    (output / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    print("Rendering figures...", flush=True)\n    make_figures(flights, causes, connections, hourly, output / "figures")\n    report = f"""# EDA 실행 결과\\n\\n- 원본: {Path(input_path).name}\\n- 기간: {quality[\'date_min\']} ~ {quality[\'date_max\']}\\n- 운항: {len(flights):,}건 / 공항: {quality[\'airports\']}개\\n- 취소: {quality[\'cancelled\']:,}건 / 회항: {quality[\'diverted\']:,}건 (서로 배타적이지 않을 수 있음)\\n- 유효 관측 연결: {len(valid):,}개\\n- 연결 지연 Pearson 상관계수: {quality[\'valid_link_pearson_correlation\']}\\n\\n## 해석 범위\\n\\n원인별 결과는 정상 도착편 중 도착 지연 15분 이상 표본이다. 결측을 0으로 대체하지 않았다. 여러 원인이 중복될 수 있다. 기체 연결은 실제 배정에 근거한 사후 관측이며 인과 효과나 실시간 예측 성능이 아니다. 공항 운항량은 수용능력 대비 혼잡도가 아니다. 실제 처리량은 검증된 비취소·비회항 운항에 한정한다. 월 경계 이전·이후 운항은 제공되지 않아 완전한 운항 체인을 보장하지 않는다.\\n\\n## 그림\\n\\n"""\n    for p in sorted((output / "figures").glob("*.png")):\n        report += f"![{p.stem}](figures/{p.name})\\n\\n"\n    (output / "SUMMARY.md").write_text(report, encoding="utf-8")\n    print(json.dumps({k: quality[k] for k in ["rows", "date_min", "date_max", "time_quality", "link_status"]}, indent=2))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Run all three BTS EDA tasks")\n    parser.add_argument("--input", type=Path, required=True)\n    parser.add_argument("--airports", type=Path, default=Path("data/reference/airports_source.csv"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--config", type=Path, default=Path("configs/eda.toml"))\n    args = parser.parse_args()\n    run(args.input, args.airports, args.output, args.config)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/flight_eda/connections.py': '"""Conservative retrospective observed sequences, not scheduled aircraft rotations."""\nimport numpy as np\nimport pandas as pd\n\n\ndef build_connections(flights, max_gap_hours=24):\n    f = flights.loc[flights.tail_num.notna()].copy()\n    # Cancelled/diverted records are retained as barriers, never silently bridged.\n    f["order_time"] = f.actual_dep_utc.where(f.actual_dep_valid).fillna(f.scheduled_dep_utc)\n    unsafe_tails = set(f.loc[f.order_time.isna(), "tail_num"])\n    f = f.sort_values(["tail_num", "order_time", "flight_id"], na_position="last")\n    f["order_tie"] = f.duplicated(["tail_num", "order_time"], keep=False)\n    fields = ["flight_id", "dest", "scheduled_arr_utc", "actual_arr_utc", "arr_delay", "time_quality_flag", "key_conflict", "order_tie"]\n    prior = f.groupby("tail_num", sort=False)[fields].shift().add_prefix("prev_")\n    f = pd.concat([f, prior], axis=1)\n    out = pd.DataFrame({\n        "prev_flight_id": f.prev_flight_id, "next_flight_id": f.flight_id, "tail_number": f.tail_num,\n        "connection_airport": f.origin, "next_scheduled_dep_utc": f.scheduled_dep_utc,\n        "prev_arr_delay_min": f.prev_arr_delay, "next_dep_delay_min": f.dep_delay,\n        "scheduled_turnaround_min": (f.scheduled_dep_utc - f.prev_scheduled_arr_utc).dt.total_seconds() / 60,\n        "actual_turnaround_min": (f.actual_dep_utc - f.prev_actual_arr_utc).dt.total_seconds() / 60,\n    })\n    out["connection_delay_change_min"] = out.next_dep_delay_min - out.prev_arr_delay_min\n    conditions = [\n        f.prev_flight_id.isna(), f.tail_num.isin(unsafe_tails),\n        f.key_conflict | f.prev_key_conflict.fillna(False).astype(bool) | f.order_tie | f.prev_order_tie.fillna(False).astype(bool),\n        f.time_quality_flag.ne("valid") | f.prev_time_quality_flag.ne("valid"),\n        f.origin.ne(f.prev_dest).fillna(True), out.actual_turnaround_min.lt(0),\n        out.scheduled_turnaround_min.lt(0), out.actual_turnaround_min.gt(max_gap_hours * 60)]\n    out["link_status"] = np.select([c.to_numpy(dtype=bool, na_value=False) for c in conditions],\n        ["boundary_unknown", "unorderable_tail", "ambiguous", "irregular_or_invalid", "airport_mismatch",\n         "time_overlap", "schedule_overlap", "long_gap"], default="valid")\n    # Nullable metrics remain available for diagnostics; only valid links enter EDA.\n    return out.reset_index(drop=True)\n\n\ndef recovery_summary(connections):\n    valid = connections.loc[connections.link_status.eq("valid")].copy()\n    valid["scheduled_turnaround_bin"] = pd.cut(valid.scheduled_turnaround_min, [0, 30, 60, 90, 120, 240, float("inf")], include_lowest=True)\n    return valid.groupby("scheduled_turnaround_bin", observed=True).agg(\n        links=("next_flight_id", "size"), mean_prev_delay=("prev_arr_delay_min", "mean"),\n        mean_next_delay=("next_dep_delay_min", "mean"), median_change=("connection_delay_change_min", "median")\n    ).reset_index()\n', 'src/flight_eda/delay_causes.py': '"""Cause minutes: zeros are observed zeros, missing values remain missing."""\nimport numpy as np\nimport pandas as pd\nfrom .preprocess import CAUSES\n\n\ndef summarize_causes(flights, group_by=()):\n    eligible = flights.loc[flights.cancelled.eq(0) & flights.diverted.eq(0) & flights.arr_delay.ge(15)]\n    groups = eligible.groupby(list(group_by), dropna=False) if group_by else [((), eligible)]\n    records = []\n    for key, group in groups:\n        if not isinstance(key, tuple):\n            key = (key,)\n        total = group[CAUSES].sum(min_count=1).sum(min_count=1)\n        for cause in CAUSES:\n            x = group[cause]\n            positive = x[x > 0]\n            minutes = x.sum(min_count=1)\n            records.append(dict(zip(group_by, key)) | {\n                "cause": cause, "eligible_flights": len(group), "observed_flights": int(x.notna().sum()),\n                "missing_flights": int(x.isna().sum()), "positive_flights": len(positive),\n                "positive_rate_of_observed": len(positive) / x.notna().sum() if x.notna().any() else np.nan,\n                "total_delay_minutes": minutes, "minute_share": minutes / total if total > 0 else np.nan,\n                "mean_positive_minutes": positive.mean(), "median_positive_minutes": positive.median(),\n                "p90_positive_minutes": positive.quantile(.9)})\n    columns = list(group_by) + ["cause", "eligible_flights", "observed_flights", "missing_flights", "positive_flights",\n                               "positive_rate_of_observed", "total_delay_minutes", "minute_share",\n                               "mean_positive_minutes", "median_positive_minutes", "p90_positive_minutes"]\n    return pd.DataFrame(records, columns=columns)\n', 'src/flight_eda/plots.py': '"""Static publication-friendly figures; no sampling in summary calculations."""\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\ndef make_figures(flights, causes, connections, hourly, output):\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})\n    fig, ax = plt.subplots(figsize=(9, 4.5))\n    causes.plot.barh(x="cause", y="total_delay_minutes", ax=ax, legend=False, color="#22577a")\n    ax.set(xlabel="Reported delay minutes (arrival delay >= 15 min)", ylabel="", title="Delay cause contributions")\n    fig.tight_layout(); fig.savefig(output / "01_delay_causes.png"); plt.close(fig)\n    valid = connections.loc[connections.link_status.eq("valid")]\n    fig, ax = plt.subplots(figsize=(7, 5))\n    if len(valid):\n        artist = ax.hexbin(valid.prev_arr_delay_min, valid.next_dep_delay_min, gridsize=45, bins="log", mincnt=1, cmap="Blues")\n        fig.colorbar(artist, ax=ax, label="Link count (log scale)")\n    ax.set(xlabel="Previous arrival delay (min)", ylabel="Next departure delay (min)", title="Observed consecutive aircraft flights")\n    fig.tight_layout(); fig.savefig(output / "02_aircraft_delay.png"); plt.close(fig)\n    volume = flights.groupby("origin").size().nlargest(15).sort_values()\n    fig, ax = plt.subplots(figsize=(9, 5))\n    volume.plot.barh(ax=ax, color="#22577a")\n    ax.set(xlabel="Scheduled departures (including cancellations)", ylabel="Airport", title="Top departure airports")\n    fig.tight_layout(); fig.savefig(output / "03_airport_volume.png"); plt.close(fig)\n    busiest = volume.index[-1] if len(volume) else None\n    subset = flights.loc[flights.origin.eq(busiest)]\n    matrix = subset.groupby(["scheduled_dep_weekday", "scheduled_dep_hour"]).size().unstack(fill_value=0).reindex(index=range(7), columns=range(24), fill_value=0)\n    fig, ax = plt.subplots(figsize=(10, 4))\n    artist = ax.imshow(matrix, aspect="auto", cmap="Blues")\n    ax.set(xticks=range(24), yticks=range(7), yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],\n           xlabel="Local scheduled departure hour", title=f"{busiest}: total scheduled departures by weekday and hour")\n    fig.colorbar(artist, ax=ax, label="Flights in observed period")\n    fig.tight_layout(); fig.savefig(output / "04_airport_heatmap.png"); plt.close(fig)\n    h = hourly.loc[hourly.airport.eq(busiest) & hourly.dep_delay_valid.gt(0)]\n    fig, ax = plt.subplots(figsize=(8, 5))\n    ax.scatter(h.dep_scheduled, h.dep_mean_delay, alpha=.35, s=12, color="#22577a")\n    ax.set(xlabel="Scheduled departures per UTC hour", ylabel="Mean departure delay (min; regular flights)",\n           title=f"{busiest}: traffic and delay (descriptive, not causal)")\n    fig.tight_layout(); fig.savefig(output / "05_traffic_delay.png"); plt.close(fig)\n', 'src/flight_eda/preprocess.py': '"""Read BTS export headers, preserve provenance and reconstruct validated UTC times."""\nfrom pathlib import Path\nimport hashlib\nimport numpy as np\nimport pandas as pd\n\nCAUSES = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]\nREQUIRED = ["fl_date", "op_unique_carrier", "op_carrier_airline_id", "op_carrier_fl_num",\n            "tail_num", "origin", "dest", "origin_airport_id", "dest_airport_id",\n            "crs_dep_time", "crs_arr_time", "crs_elapsed_time", "dep_time", "arr_time",\n            "dep_delay", "arr_delay", "actual_elapsed_time", "cancelled", "diverted",\n            "taxi_out", "taxi_in"] + CAUSES\nNUMERIC = ["dep_delay", "arr_delay", "crs_elapsed_time", "actual_elapsed_time",\n           "cancelled", "diverted", "taxi_out", "taxi_in"] + CAUSES\n\n\ndef file_hash(path):\n    with open(path, "rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef hhmm_minutes(values):\n    n = pd.to_numeric(values, errors="coerce")\n    valid = n.notna() & n.eq(n.round()) & n.ge(0) & n.le(2400) & (n.mod(100).lt(60))\n    return ((n // 100) * 60 + n.mod(100)).where(valid)\n\n\ndef localize_by_zone(local, zones):\n    result = pd.Series(pd.NaT, index=local.index, dtype="datetime64[ns, UTC]")\n    for zone, indexes in zones.dropna().groupby(zones.dropna()).groups.items():\n        result.loc[indexes] = local.loc[indexes].dt.tz_localize(\n            str(zone), ambiguous="NaT", nonexistent="NaT").dt.tz_convert("UTC")\n    return result\n\n\ndef clock_matches(utc, zones, raw):\n    minute = pd.Series(np.nan, index=utc.index)\n    for zone, indexes in zones.dropna().groupby(zones.dropna()).groups.items():\n        local = utc.loc[indexes].dt.tz_convert(str(zone))\n        minute.loc[indexes] = local.dt.hour * 60 + local.dt.minute\n    return minute.eq(hhmm_minutes(raw).mod(1440)).fillna(False).astype(bool)\n\n\ndef read_flights(path):\n    frame = pd.read_csv(path, dtype="string", encoding="utf-8-sig")\n    frame.columns = frame.columns.str.strip().str.lower()\n    missing = set(REQUIRED) - set(frame.columns)\n    if missing:\n        raise ValueError(f"Missing required BTS export columns: {sorted(missing)}")\n    # Original remains untouched. Exact duplicates are counted before collapsing.\n    duplicates = int(frame.duplicated().sum())\n    frame["source_row"] = np.arange(2, len(frame) + 2)\n    frame = frame.drop_duplicates(subset=[c for c in frame if c != "source_row"]).reset_index(drop=True)\n    frame["source_file"] = Path(path).name\n    for c in NUMERIC:\n        frame[c] = pd.to_numeric(frame[c], errors="coerce").astype(float)\n    for c in ["origin", "dest", "tail_num"]:\n        frame[c] = frame[c].str.strip().str.upper().replace({"": pd.NA})\n    frame["flight_date"] = pd.to_datetime(frame.fl_date.str.split().str[0], format="%m/%d/%Y", errors="coerce")\n    iso = pd.to_datetime(frame.fl_date.str[:10], format="%Y-%m-%d", errors="coerce")\n    frame["flight_date"] = frame.flight_date.fillna(iso)\n    key = ["fl_date", "op_carrier_airline_id", "op_carrier_fl_num", "origin", "dest", "crs_dep_time"]\n    frame["key_conflict"] = frame.duplicated(key, keep=False)\n    # Content-derived ID, independent of source file path or row ordering.\n    frame["flight_id"] = pd.util.hash_pandas_object(frame[key].fillna("<NA>"), index=False).map(lambda n: f"{n:016x}")\n    # Conflicting natural keys remain inspectable and are not linked.\n    conflict = frame.key_conflict\n    frame.loc[conflict, "flight_id"] = frame.loc[conflict, "flight_id"] + "-row" + frame.loc[conflict, "source_row"].astype(str)\n    frame["block_delay_recovery_min"] = (frame.dep_delay - frame.arr_delay).where(frame.cancelled.eq(0) & frame.diverted.eq(0))\n    return frame, duplicates\n\n\ndef add_times(frame, airport_reference):\n    frame = frame.copy()\n    ref = pd.read_csv(airport_reference, dtype="string")\n    if not {"iata", "tz"}.issubset(ref):\n        raise ValueError("Airport reference must have iata,tz columns")\n    ref = ref.dropna(subset=["iata", "tz"])\n    inconsistent = ref.groupby("iata").tz.nunique()\n    if (inconsistent > 1).any():\n        raise ValueError("Conflicting timezone assignments in airport reference")\n    lookup = ref.drop_duplicates("iata").set_index("iata").tz\n    frame["origin_timezone"] = frame.origin.map(lookup)\n    frame["dest_timezone"] = frame.dest.map(lookup)\n    local_dep = frame.flight_date + pd.to_timedelta(hhmm_minutes(frame.crs_dep_time), unit="m")\n    frame["scheduled_dep_utc"] = localize_by_zone(local_dep, frame.origin_timezone)\n    frame["scheduled_arr_utc"] = frame.scheduled_dep_utc + pd.to_timedelta(frame.crs_elapsed_time, unit="m")\n    frame["actual_dep_utc"] = frame.scheduled_dep_utc + pd.to_timedelta(frame.dep_delay, unit="m")\n    frame["actual_arr_utc"] = frame.actual_dep_utc + pd.to_timedelta(frame.actual_elapsed_time, unit="m")\n    frame["schedule_time_valid"] = (clock_matches(frame.scheduled_arr_utc, frame.dest_timezone, frame.crs_arr_time)\n                                    & frame.crs_elapsed_time.gt(0))\n    dep_ok = clock_matches(frame.actual_dep_utc, frame.origin_timezone, frame.dep_time)\n    arr_ok = clock_matches(frame.actual_arr_utc, frame.dest_timezone, frame.arr_time)\n    residual = (frame.actual_arr_utc - frame.scheduled_arr_utc).dt.total_seconds() / 60 - frame.arr_delay\n    frame["actual_dep_valid"] = dep_ok & frame.scheduled_dep_utc.notna()\n    frame["actual_arr_valid"] = arr_ok & residual.abs().le(1) & frame.actual_elapsed_time.gt(0)\n    frame["time_quality_flag"] = np.select(\n        [frame.origin_timezone.isna() | frame.dest_timezone.isna(), frame.cancelled.eq(1), frame.diverted.eq(1),\n         ~frame.schedule_time_valid, ~(frame.actual_dep_valid & frame.actual_arr_valid)],\n        ["missing_timezone", "cancelled", "diverted", "schedule_invalid", "actual_missing_or_inconsistent"], default="valid")\n    frame["scheduled_dep_local"] = local_dep\n    frame["scheduled_dep_hour"] = local_dep.dt.hour\n    frame["scheduled_dep_weekday"] = local_dep.dt.dayofweek\n    return frame\n\n\ndef diversion_events(frame):\n    parts = []\n    for seq in range(1, 6):\n        prefix = f"div{seq}_"\n        cols = [c for c in frame if c.startswith(prefix)]\n        airport = prefix + "airport"\n        if airport not in frame:\n            continue\n        part = frame.loc[frame[airport].notna(), ["flight_id"] + cols].copy()\n        part = part.rename(columns={c: c.removeprefix(prefix) for c in cols})\n        part["diversion_sequence"] = seq\n        parts.append(part)\n    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["flight_id", "diversion_sequence"])\n', 'src/flight_eda/__init__.py': '"""Descriptive EDA only; actual aircraft assignments are retrospective."""\n', 'configs/eda.toml': '[analysis]\ndelay_threshold_min = 15\nmax_connection_gap_hours = 24\n# A long-gap sensitivity check is also saved for 12, 24 and 48 hours.\n'}
for relative, content in SOURCE_FILES.items():
    target = ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
print('분석 모듈 준비 완료')

## 4. 공항 시간대 기준정보

2026년 1월 분석에 사용한 과거 스냅샷을 고정합니다. 최신 목록은 당시 공항 코드와 다를 수 있습니다. 다른 시기 데이터에는 기준정보의 적용 가능성을 다시 확인하세요.

In [ ]:
REFERENCE = {'url': 'https://raw.githubusercontent.com/mborsetti/airportsdata/67811dc0410629eeab7bf060568dae4287bd37ba/airportsdata/airports.csv', 'commit': '67811dc0410629eeab7bf060568dae4287bd37ba', 'as_of': '2026-02-01', 'retrieved_utc': '2026-11-15T16:26:46.362523+00:00', 'sha256': 'a4c18cfcb56cebdf55d4e8fbd4c49425aa5f76eb51dc5aca6f6b5d1aeb6dc2a6', 'license_url': 'https://github.com/mborsetti/airportsdata/blob/67811dc0410629eeab7bf060568dae4287bd37ba/LICENSE'}
AIRPORTS = ROOT / 'data/reference/airports_source.csv'
AIRPORTS.parent.mkdir(parents=True, exist_ok=True)
if not AIRPORTS.exists() or hashlib.sha256(AIRPORTS.read_bytes()).hexdigest() != REFERENCE['sha256']:
    with urllib.request.urlopen(REFERENCE['url'], timeout=120) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != REFERENCE['sha256']:
        raise ValueError('공항 기준정보 해시가 다릅니다. 원본을 확인하세요.')
    AIRPORTS.write_bytes(content)
(AIRPORTS.parent / 'source.json').write_text(json.dumps(REFERENCE, indent=2), encoding='utf-8')
print('공항 기준정보 검증 완료:', REFERENCE['commit'])

## 5. 전체 파이프라인 실행

전체 CSV를 처리합니다. 재실행 시 새로운 결과 폴더를 만들며 기존 결과는 덮어쓰지 않습니다. 처리 중 단계별 진행 상황이 출력됩니다.

In [ ]:
import os
RUN = ROOT / 'reports/runs' / datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT / 'src')
subprocess.run([sys.executable, '-u', '-m', 'flight_eda.cli',
    '--input', str(INPUT), '--airports', str(AIRPORTS),
    '--config', str(ROOT / 'configs/eda.toml'), '--output', str(RUN)],
    cwd=ROOT, env=env, check=True)
print('결과 폴더:', RUN)

## 6. 분석 결과 탐색

먼저 품질·제외 사유를 확인한 뒤 세 가지 분석을 순서대로 진행합니다. 아래 분석은 완료된 산출물만 읽습니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
plt.rcParams['figure.figsize'] = (10, 5)

### 데이터 품질과 분석 모집단

결측·제외 사유와 분모를 먼저 확인합니다. 사후 관측 EDA이며 인과 효과나 예측 성능을 의미하지 않습니다. 정의와 한계는 README를 참고하세요.

In [ ]:
quality = json.loads((RUN / "quality_report.json").read_text(encoding="utf-8"))
{k: v for k, v in quality.items() if k != "column_missing_counts"}

In [ ]:
pd.Series(quality["column_missing_counts"]).sort_values(ascending=False).head(30)

In [ ]:
pd.Series(quality["time_quality"]).plot.barh(title="Time reconstruction quality"); plt.tight_layout(); plt.show()

In [ ]:
pd.Series(quality["link_status"]).sort_values().plot.barh(title="Observed sequence link status"); plt.tight_layout(); plt.show()

### 지연 원인별 분포 및 특성

결측·제외 사유와 분모를 먼저 확인합니다. 사후 관측 EDA이며 인과 효과나 예측 성능을 의미하지 않습니다. 정의와 한계는 README를 참고하세요.

In [ ]:
causes = pd.read_csv(RUN / "delay_causes.csv")
causes

In [ ]:
causes.plot.barh(x="cause", y="total_delay_minutes", legend=False, title="Reported delay minutes"); plt.tight_layout(); plt.show()

In [ ]:
carrier = pd.read_csv(RUN / "delay_causes_by_carrier.csv")
carrier.pivot(index="op_unique_carrier", columns="cause", values="minute_share").plot.bar(stacked=True, figsize=(12, 5)); plt.ylabel("Share of reported cause minutes"); plt.tight_layout(); plt.show()

In [ ]:
pd.read_csv(RUN / "delay_causes_by_airport.csv").sort_values("total_delay_minutes", ascending=False).head(20)

In [ ]:
pd.read_csv(RUN / "delay_causes_by_hour.csv").pivot(index="scheduled_dep_hour", columns="cause", values="mean_positive_minutes").plot(figsize=(10, 5)); plt.ylabel("Mean positive cause minutes"); plt.tight_layout(); plt.show()

In [ ]:
pd.read_csv(RUN / "cancellations.csv")

### 동일 항공기 연속 운항 및 지연 패턴

결측·제외 사유와 분모를 먼저 확인합니다. 사후 관측 EDA이며 인과 효과나 예측 성능을 의미하지 않습니다. 정의와 한계는 README를 참고하세요.

In [ ]:
links = pd.read_parquet(RUN / "flight_connections.parquet")
valid = links.loc[links.link_status.eq("valid")]
valid[["prev_arr_delay_min", "next_dep_delay_min", "scheduled_turnaround_min", "actual_turnaround_min"]].describe()

In [ ]:
plt.hexbin(valid.prev_arr_delay_min, valid.next_dep_delay_min, gridsize=50, bins="log", mincnt=1); plt.colorbar(label="Link count (log scale)"); plt.xlabel("Previous arrival delay (min)"); plt.ylabel("Next departure delay (min)"); plt.show()

In [ ]:
pd.read_csv(RUN / "connection_recovery.csv")

In [ ]:
pd.read_csv(RUN / "connection_sensitivity.csv")

In [ ]:
tail = valid.groupby("tail_number").size().idxmax()
example = valid.loc[valid.tail_number.eq(tail)].sort_values("next_scheduled_dep_utc")
plt.figure(figsize=(12, 4)); plt.plot(example.next_scheduled_dep_utc, example.next_dep_delay_min, label="Next departure delay"); plt.plot(example.next_scheduled_dep_utc, example.prev_arr_delay_min, label="Previous arrival delay"); plt.title(f"Illustrative observed tail: {tail} (most valid links)"); plt.ylabel("Minutes"); plt.legend(); plt.tight_layout(); plt.show()

### 공항 운항량과 혼잡 징후

결측·제외 사유와 분모를 먼저 확인합니다. 사후 관측 EDA이며 인과 효과나 예측 성능을 의미하지 않습니다. 정의와 한계는 README를 참고하세요.

In [ ]:
hourly = pd.read_parquet(RUN / "airport_hourly.parquet")
volume = hourly.groupby("airport")[["dep_scheduled", "arr_scheduled", "dep_actual_regular", "arr_actual_regular"]].sum().sort_values("dep_scheduled", ascending=False)
volume.head(20)

In [ ]:
volume.head(15).dep_scheduled.sort_values().plot.barh(title="Scheduled departures"); plt.tight_layout(); plt.show()

In [ ]:
airport = volume.index[0]
h = hourly.loc[hourly.airport.eq(airport) & hourly.dep_delay_valid.gt(0)]
plt.scatter(h.dep_scheduled, h.dep_mean_delay, alpha=.3); plt.xlabel("Scheduled departures per UTC hour"); plt.ylabel("Mean departure delay (min)"); plt.title(airport); plt.show()

In [ ]:
plt.scatter(h.dep_scheduled, h.dep_mean_taxi, alpha=.3); plt.xlabel("Scheduled departures per UTC hour"); plt.ylabel("Mean taxi-out (min)"); plt.title(airport); plt.show()

In [ ]:
flights = pd.read_parquet(RUN / "flights.parquet", columns=["origin", "scheduled_dep_hour", "scheduled_dep_weekday"])
pivot = flights.loc[flights.origin.eq(airport)].groupby(["scheduled_dep_weekday", "scheduled_dep_hour"]).size().unstack(fill_value=0).reindex(index=range(7), columns=range(24), fill_value=0)
plt.figure(figsize=(12, 4)); plt.imshow(pivot, aspect="auto"); plt.colorbar(label="Flights in observed period"); plt.xlabel("Local hour"); plt.ylabel("Weekday (Monday=0)"); plt.show()

## 7. 해석 범위

- 원인 분석은 비취소·비회항 중 도착 지연 15분 이상 집단입니다. 원인값 결측과 0을 구분하며 복수 원인이 가능합니다.
- 실제 기체 배정에 따른 사후 관측 연결입니다. 상관관계와 지연 변화량은 인과적인 전파량이 아닙니다.
- 취소·회항 및 불확실한 연결은 제외 사유를 보존합니다. 회항 상세 경로 복원은 포함하지 않았습니다.
- 공항 운항량은 수용능력 대비 혼잡도 자체가 아닙니다. 실제 처리량은 검증된 비취소·비회항 운항만 포함합니다.
- 한 달 자료로 계절성을 결론 내릴 수 없고, 월 경계의 선행·후속 운항은 누락될 수 있습니다.

## 8. 결과 저장 및 다운로드

ZIP에는 분석 결과와 실행 코드, 공항 기준정보 출처를 담습니다. 원본 CSV는 포함하지 않습니다. Drive 입력 방식에서는 결과를 Drive에도 복사할 수 있습니다.

In [ ]:
INCLUDE_PARQUET = True #@param {type:"boolean"}
SAVE_TO_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/flight_eda_results' #@param {type:"string"}
import zipfile
archive = ROOT / f'flight-eda-results-{RUN.name}.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in RUN.rglob('*'):
        if p.is_file() and (INCLUDE_PARQUET or p.suffix != '.parquet'):
            z.write(p, Path('results') / p.relative_to(RUN))
    for relative in SOURCE_FILES:
        z.write(ROOT / relative, Path('code') / relative)
    z.write(AIRPORTS.parent / 'source.json', 'code/data/reference/source.json')
if SAVE_TO_DRIVE:
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    destination = Path(DRIVE_OUTPUT_DIR)
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, destination / archive.name)
    print('Drive 저장:', destination / archive.name)
from google.colab import files
files.download(str(archive))